# Toto 2.0 — DIMER zero-shot forecasting tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/toto-forecasting-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/toto-forecasting-pipeline/blob/main/tutorials/toto_forecasting_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Datadog%2FToto--2.0--2.5B-ffcc4d?style=flat)](https://huggingface.co/Datadog/Toto-2.0-2.5B) [![Upstream](https://img.shields.io/badge/Upstream-DataDog%2Ftoto-181717?style=flat&logo=github&logoColor=white)](https://github.com/DataDog/toto) [![arXiv](https://img.shields.io/badge/arXiv-2605.20119-b31b1b.svg)](https://arxiv.org/abs/2605.20119)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** zero-shot multivariate probabilistic time-series forecasting with chronological evaluation, using the pinned `Datadog/Toto-2.0-2.5B` checkpoint

**This notebook is standalone.** It carries the repository's package (3 modules under `src/toto_forecasting_pipeline/`, at revision `e2708a20e541`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `51a2812bbe449437c01b79c0e425ed578f335f5b` (~9817 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, obtains the tutorial sample automatically, validates it into an input manifest before the model runs, runs the task locally in this kernel, writes the evaluation report, and exports machine-readable outputs with provenance. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in the sample cell and re-run from that cell to supply your own input. It passes through the same notebook-local validation, task, evaluation-report and export cells as the sample; the expected input format, the ceilings and the privacy guidance are stated in the Prerequisites and in the sample cell, and the upload stays inside this runtime. BYOD is optional and never part of the default path.

At inference Toto 2.0 consumes the context of one or more variates in 32-step patches and decodes nine forecast quantiles (q=0.1 … q=0.9) per step of the requested horizon, in blocks of `decode_block_size` steps (768 here; `None` requests a single forward-pass decode); the pipeline reports q=0.5 as the point forecast (a model median, not a mean) and keeps all nine quantiles. **No adaptation occurs:** no gradient training, fine-tuning, exogenous-variable conditioning or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights and the model configuration, and the carried package adds snapshot verification, the input contract (`validate_target`, `validate_horizon`, `decode_block_size` and patch-multiple rules, mask-aware left padding), the `mae` / `rmse` / `last_value_baseline` / `interval_coverage` helpers and the public `validate_inputs` / `evaluation_report` stages. The default sample is a deterministic two-variate synthetic series generated in code and scored on one chronological holdout; its metrics are demonstration (plumbing) evidence, not a benchmark claim.

**Learning objectives:** install the pinned runtime, read what the carried package guarantees, resolve and digest-verify the immutable upstream checkpoint (9.8 GB SafeTensors), generate a deterministic synthetic series (or upload a CSV), make a leakage-safe chronological holdout, validate the context into an input manifest, run the zero-shot forecast on a CUDA GPU, read the median and the model quantiles correctly, produce an evaluation report that compares MAE/RMSE with a naive last-value baseline and reports empirical q10–q90 coverage, and export machine-readable forecasts plus provenance.

**This notebook does not demonstrate:** gradient training or fine-tuning, exogenous/covariate conditioning (not part of the Toto 2.0 open inference release), missing-value imputation (non-finite targets are rejected), anomaly detection, or calibrated prediction intervals. The quantiles are model quantiles, not guaranteed coverage; a single holdout is not a deployment-variability estimate.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12) with a **CUDA GPU of at least 16 GB** — the 2.5B checkpoint is loaded in its stored float32 precision (9.8 GB of weights) and the pipeline refuses `device='cuda'` when no CUDA device is present; the CPU path is not the supported release-reference path. The pinned `torch==2.7.0` install and the 9.8 GB checkpoint fetch are the largest downloads of the run.
- **Knowledge:** basic Python and NumPy; what a forecast horizon, a context window and a quantile forecast are.
- **Data:** the default sample is a deterministic two-variate trend + seasonal series (320 steps, fixed seed) generated in code, so nothing else is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one UTF-8 CSV with a unique `timestamp` column and one or more finite numeric target columns in chronological order, at least 80 rows (32 context + 48 holdout); missing target values are rejected by this DIMER contract rather than silently imputed. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `Datadog/Toto-2.0-2.5B` snapshot (~9817 MB in total) at revision `51a2812bbe44…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `numpy`, `pandas` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'toto-2==2.0.0',
    'torch==2.7.0',
    'torchaudio==2.7.0',
    'torchvision==0.22.0',
    'huggingface-hub==0.36.2',
    'transformers==4.57.6',
    'numpy==1.26.4',
    'pandas==2.2.3',
]
NOTEBOOK_SOURCE = {
    'repository': 'toto-forecasting-pipeline',
    'repository_revision': 'e2708a20e54170d1705d35c96c4a3e6e07f51dd4',
    'embedded_module': 'src/toto_forecasting_pipeline/pipeline.py',
    'embedded_modules': ['src/toto_forecasting_pipeline/evaluation.py', 'src/toto_forecasting_pipeline/validation.py', 'src/toto_forecasting_pipeline/pipeline.py'],
    'module_sha256': 'e491e5ad4c4f5ea1a89d1a1278bc30a942c72909f7ffd184dfa6beb408cd54cb',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, numpy, pandas
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'numpy': numpy.__version__, 'pandas': pandas.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/toto_forecasting_pipeline/` @ `e2708a20e541`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/toto_forecasting_pipeline/evaluation.py`

In [ ]:
from __future__ import annotations

import numpy as np


def mae(y_true, y_pred) -> float:
    actual = np.asarray(y_true, dtype=float)
    predicted = np.asarray(y_pred, dtype=float)
    if actual.shape != predicted.shape:
        raise ValueError("y_true and y_pred must have the same shape")
    return float(np.mean(np.abs(actual - predicted)))


def rmse(y_true, y_pred) -> float:
    actual = np.asarray(y_true, dtype=float)
    predicted = np.asarray(y_pred, dtype=float)
    if actual.shape != predicted.shape:
        raise ValueError("y_true and y_pred must have the same shape")
    return float(np.sqrt(np.mean((actual - predicted) ** 2)))


def last_value_baseline(context, horizon: int) -> np.ndarray:
    values = np.asarray(context, dtype=float)
    if values.ndim == 1:
        values = values[None, :]
    if values.ndim != 2 or values.shape[1] < 1:
        raise ValueError(
            "context must have shape (variates, time) with at least one observation"
        )
    return np.repeat(values[:, -1:], horizon, axis=1)


def interval_coverage(y_true, lower, upper) -> float:
    actual = np.asarray(y_true, dtype=float)
    lower_bound = np.asarray(lower, dtype=float)
    upper_bound = np.asarray(upper, dtype=float)
    if not (actual.shape == lower_bound.shape == upper_bound.shape):
        raise ValueError("coverage arrays must share shape")
    return float(np.mean((actual >= lower_bound) & (actual <= upper_bound)))

**Module 2/3:** `src/toto_forecasting_pipeline/validation.py` (carried verbatim; see the note above)

In [ ]:
from __future__ import annotations

import numpy as np

MIN_CONTEXT = 32  # observations; shorter targets are rejected before any model work
MAX_CONTEXT = 16_384  # observations per variate
MAX_HORIZON = 4096  # forecast steps per call


def validate_target(target, *, min_context: int = MIN_CONTEXT, max_context: int = MAX_CONTEXT) -> np.ndarray:
    values = np.asarray(target, dtype=np.float32)
    if values.ndim == 1:
        values = values[None, :]
    if values.ndim != 2:
        raise ValueError("target must be 1D or 2D with shape (variates, time)")
    if not min_context <= values.shape[1] <= max_context:
        raise ValueError(
            f"context length must be between {min_context} and {max_context}"
        )
    if not np.isfinite(values).all():
        raise ValueError("target must contain only finite values")
    return values


def validate_horizon(horizon: int, *, max_horizon: int = MAX_HORIZON) -> int:
    if not isinstance(horizon, int) or not 1 <= horizon <= max_horizon:
        raise ValueError(f"horizon must be an integer from 1 to {max_horizon}")
    return horizon

**Module 3/3:** `src/toto_forecasting_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""Zero-shot probabilistic forecasting with the pinned ``Datadog/Toto-2.0-2.5B`` checkpoint.

The class loads the checkpoint only from a digest-verified local snapshot (``weights/<key>/``:
``config.json`` + ``model.safetensors``) or, when explicitly allowed, from the Hugging Face Hub at
the pinned revision. Model construction happens inside the upstream ``toto2`` package from its own
``Toto2ModelConfig``; the weights are SafeTensors, and no model-repository code is executed.
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np

# standalone rewrite (build_notebook.py): `from .evaluation import interval_coverage, last_value_baseline, mae, rmse` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .validation import MAX_CONTEXT, MAX_HORIZON, MIN_CONTEXT, validate_horizon, validate_target` removed — names are kernel globals defined by the carried modules

MODEL_ID = "Datadog/Toto-2.0-2.5B"
MODEL_REVISION = "51a2812bbe449437c01b79c0e425ed578f335f5b"
MODEL_LICENSE = "Apache-2.0"
MODEL_KEY = "toto-2.0-2.5b"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
CONFIG_FILE = "config.json"
WEIGHTS_FILE = "model.safetensors"
QUANTILES = (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9)
MEDIAN_INDEX = 4  # QUANTILES[4] == 0.5: the point forecast is the model median, not a mean
POINT_FORECAST = "median (q=0.5)"
DEFAULT_DECODE_BLOCK_SIZE = 768


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path or DEFAULT_WEIGHTS_DIR)
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the 9.8 GB checkpoint). Returns the relative paths fetched; `verify_snapshot` still runs
    after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _validate_decode_block_size(value: int | None) -> int | None:
    if value is None:
        return None
    if not isinstance(value, int) or isinstance(value, bool) or value < 1:
        raise ValueError("decode_block_size must be None or a positive integer")
    return value


INPUT_SCHEMA: dict[str, Any] = {
    "input": "1D array (time) or 2D array (variates, time) of finite numbers, cast to float32",
    "context_length": [MIN_CONTEXT, MAX_CONTEXT],
    "horizon": [1, MAX_HORIZON],
    "decode_block_size": "None (single forward-pass decode) or a positive multiple of the model patch size",
    "model_dependent_checks": (
        "forecast() additionally requires context_length >= patch_size and "
        "decode_block_size % patch_size == 0, "
        "read from the loaded model config (32 for the pinned checkpoint)"
    ),
    "preprocessing": (
        "context left-padded to the next patch multiple with masked (unobserved) positions; "
        "upstream toto2 scales the context causally inside the model"
    ),
}


def _check_inputs(target: Any, horizon: int, decode_block_size: int | None) -> tuple[np.ndarray, int | None]:
    """Raise ValueError naming the first violated model-independent ceiling; return the validated inputs."""
    values = validate_target(target)
    validate_horizon(horizon)
    return values, _validate_decode_block_size(decode_block_size)


def validate_inputs(
    target: Any,
    *,
    horizon: int,
    decode_block_size: int | None = DEFAULT_DECODE_BLOCK_SIZE,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-variate observations, verdict).

    Rejection is reported by raising exactly as ``forecast`` would for the model-independent checks
    (target shape and finiteness, context length, horizon, ``decode_block_size`` type); the two
    patch-size checks need the loaded model and stay in ``forecast``. A caller that wants a finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    values, block = _check_inputs(target, horizon, decode_block_size)
    if names is not None and len(names) != values.shape[0]:
        raise ValueError("names must have one entry per variate")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {
                "id": names[i] if names else f"variate-{i}",
                "context_length": int(values.shape[1]),
                "min": float(values[i].min()),
                "max": float(values[i].max()),
            }
            for i in range(values.shape[0])
        ],
        "horizon": horizon,
        "decode_block_size": block,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any],
    truth: Any = None,
    *,
    context: Any = None,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``truth`` (the withheld future, shape ``(variates, horizon)``) the report carries the
    repository's ``mae`` / ``rmse`` on the median forecast, ``interval_coverage`` of the q10–q90 band,
    and — when ``context`` is supplied — the same ``mae`` / ``rmse`` for ``last_value_baseline``, with
    verdict ``sample-sanity``. Without ``truth`` the verdict is ``not-measurable``.
    """
    median = np.asarray(result["median"], dtype=float)
    base = {
        "task": "zero-shot probabilistic time-series forecasting",
        "point_forecast": POINT_FORECAST,
        "score_semantics": (
            "quantile levels 0.1..0.9 are model quantiles, not calibrated confidence intervals"
        ),
        "sample_kind": sample_kind,
        "horizon": int(result["horizon"]),
        "context_length": int(result["context_length"]),
        "n_variates": int(median.shape[0]),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if truth is None:
        return {
            **base,
            "metrics": [],
            "baselines": [],
            "verdict": "not-measurable",
            "reason": "no withheld future values were supplied for the forecast horizon",
            "needs": (
                "a chronological holdout: withhold the final `horizon` observations of the target, "
                "forecast from the remaining context, and score the median with mae/rmse against them "
                "and against the last_value_baseline, repeated over representative periods"
            ),
        }
    actual = np.asarray(truth, dtype=float)
    if actual.ndim == 1:
        actual = actual[None, :]
    if actual.shape != median.shape:
        raise ValueError(f"truth shape {actual.shape} != median shape {median.shape}")
    estimation = "single chronological holdout, no dispersion estimate"
    metrics = [
        {"id": "mae", "value": mae(actual, median), "estimation": estimation},
        {"id": "rmse", "value": rmse(actual, median), "estimation": estimation},
    ]
    quantiles = np.asarray(result["quantiles"], dtype=float)
    levels = list(result["quantile_levels"])
    if quantiles.shape[:1] == actual.shape[:1] and 0.1 in levels and 0.9 in levels:
        metrics.append(
            {
                "id": "interval_coverage",
                "band": [0.1, 0.9],
                "value": interval_coverage(
                    actual, quantiles[:, levels.index(0.1), :], quantiles[:, levels.index(0.9), :]
                ),
                "nominal": 0.8,
                "estimation": estimation,
            }
        )
    baselines = []
    if context is not None:
        baseline = last_value_baseline(context, int(result["horizon"]))
        baselines.append(
            {
                "id": "last_value_baseline",
                "metrics": [
                    {"id": "mae", "value": mae(actual, baseline)},
                    {"id": "rmse", "value": rmse(actual, baseline)},
                ],
            }
        )
    return {
        **base,
        "metrics": metrics,
        "baselines": baselines,
        "verdict": "sample-sanity",
        "reason": (
            f"one chronological holdout of {actual.shape[1]} step(s) on the tutorial sample; not a benchmark"
        ),
        "needs": (
            "repeated holdouts over representative periods of the deployment series for any "
            "generalisable claim"
        ),
    }


@dataclass
class TotoForecastPipeline:
    _model: Any
    device: str
    source: str = "injected"

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> TotoForecastPipeline:
        import torch
        from toto2 import Toto2Model

        resolved_device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        if resolved_device.startswith("cuda") and not torch.cuda.is_available():
            raise RuntimeError(
                "device='cuda' was requested but no CUDA device is available; the 2.5B checkpoint's "
                "release-reference path is a CUDA GPU with at least 16 GB of memory"
            )
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            # A directory argument makes toto2 build Toto2ModelConfig from config.json and load
            # model.safetensors from that directory (no Hub resolution, no cache lookup).
            model = Toto2Model.from_pretrained(str(root), map_location="cpu")
            source = "local-snapshot"
        elif allow_download:
            model = Toto2Model.from_pretrained(
                MODEL_ID,
                revision=MODEL_REVISION,
                map_location="cpu",
            )
            source = "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        model = model.to(resolved_device).eval()
        return cls(model, resolved_device, source)

    def forecast(
        self,
        target,
        *,
        horizon: int,
        decode_block_size: int | None = DEFAULT_DECODE_BLOCK_SIZE,
    ) -> dict[str, Any]:
        values, decode_block_size = _check_inputs(target, horizon, decode_block_size)
        patch_size = int(self._model.config.patch_size)
        if decode_block_size is not None and decode_block_size % patch_size:
            raise ValueError(
                f"decode_block_size must be a multiple of the model patch size ({patch_size})"
            )
        # Upstream patches the context in blocks of `patch_size` and requires the context length
        # to be a multiple of it. Pad on the left and mark the padded positions unobserved: the
        # upstream causal scaler and patch embedding are mask-aware, so zero pads carry no signal.
        # (Upstream's own GluonTS adapter truncates to a patch multiple instead; padding keeps every
        # observed value.) At least one full patch of real context is required.
        if values.shape[1] < patch_size:
            raise ValueError(f"context length must be at least the model patch size ({patch_size})")
        context_padding = (patch_size - values.shape[1] % patch_size) % patch_size

        import torch

        target_tensor = torch.as_tensor(
            values,
            dtype=torch.float32,
            device=self.device,
        ).unsqueeze(0)
        target_mask = torch.ones_like(target_tensor, dtype=torch.bool)
        if context_padding:
            pad = (context_padding, 0)
            target_tensor = torch.nn.functional.pad(target_tensor, pad, value=0.0)
            target_mask = torch.nn.functional.pad(target_mask, pad, value=False)
        series_ids = torch.arange(
            values.shape[0],
            device=self.device,
            dtype=torch.long,
        ).unsqueeze(0)
        with torch.inference_mode():
            forecast = self._model.forecast(
                {
                    "target": target_tensor,
                    "target_mask": target_mask,
                    "series_ids": series_ids,
                },
                horizon=horizon,
                decode_block_size=decode_block_size,
                has_missing_values=context_padding > 0,
            )
        quantiles = np.asarray(forecast.detach().cpu(), dtype=float)
        expected_shape = (len(QUANTILES), 1, values.shape[0], horizon)
        if quantiles.shape != expected_shape:
            raise RuntimeError(
                f"unexpected Toto forecast shape: {quantiles.shape}; "
                f"expected {expected_shape}"
            )
        quantiles = np.transpose(quantiles[:, 0, :, :], (1, 0, 2))
        return {
            "quantiles": quantiles,
            "quantile_levels": QUANTILES,
            "median": quantiles[:, MEDIAN_INDEX, :],
            "point_forecast": POINT_FORECAST,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "horizon": horizon,
            "context_length": values.shape[1],
            "context_padding": context_padding,
            "patch_size": patch_size,
            "n_variates": values.shape[0],
            "decode_block_size": decode_block_size,
            "device": self.device,
            "source": self.source,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `3`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `51a2812bbe44…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `TotoForecastPipeline.from_pretrained(device='cuda', weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "toto-2.0-2.5b",
  "modelId": "Datadog/Toto-2.0-2.5B",
  "revision": "51a2812bbe449437c01b79c0e425ed578f335f5b",
  "files": [
    {
      "path": "README.md",
      "bytes": 7188,
      "sha256": "5cd75f3d08d1574c085ee31a2886855d82247f76cb3bd0b8015750a7fc1d316d"
    },
    {
      "path": "config.json",
      "bytes": 598,
      "sha256": "172904c65bb5af77a95e1a81cdd9214f4c9d983651d6dceef1375d3d09f47a36"
    },
    {
      "path": "model.safetensors",
      "bytes": 9817176960,
      "sha256": "dc08942b20751ac906167194d4ca4aa06b4367e80aabe5f1d5153b30b874bdb9"
    }
  ],
  "totalBytes": 9817184746
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = TotoForecastPipeline.from_pretrained(device='cuda', weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Generate the synthetic sample or optional BYOD

The default sample is **synthetic**: 320 steps of two variates — a linear trend plus a sine season plus small Gaussian noise, and a second variate coupled to the first plus a cosine season — from a fixed seed (`np.random.default_rng(11)`), so it needs no download and its SHA-256 is printed for the record. It has a real future — the final `HORIZON` steps are withheld in the next section — so the evaluation report can score the forecast, but a synthetic series says nothing about any deployment domain. BYOD is optional and disabled by default; when enabled, the raw CSV header is inspected before pandas reads the file so duplicate column names cannot be silently renamed, timestamps must parse, increase strictly and be regularly spaced, and the series must keep at least `MIN_CONTEXT` observations after the holdout is withheld. Look for a dictionary naming the sample kind, its shape and digest.

In [ ]:
import csv
import hashlib

import pandas as pd

USE_BYOD = False  # @param {type:"boolean"}
HORIZON = 48
DECODE_BLOCK_SIZE = 768

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    sample_name = next(iter(uploaded))
    with open(sample_name, newline='', encoding='utf-8-sig') as handle:
        header = next(csv.reader(handle), [])
    if not header or len(header) != len(set(header)):
        raise ValueError('CSV must have non-empty unique column names; duplicate headers are rejected before pandas ingestion')
    if 'timestamp' not in header:
        raise ValueError('BYOD CSV must contain a timestamp column')
    variate_names = [column for column in header if column != 'timestamp']
    if not variate_names:
        raise ValueError('BYOD CSV must contain at least one numeric target column')
    frame = pd.read_csv(sample_name)
    timestamps = pd.to_datetime(frame['timestamp'], errors='raise')
    if not timestamps.is_monotonic_increasing or timestamps.duplicated().any():
        raise ValueError('timestamps must be unique and strictly increasing')
    if len(timestamps) > 2 and timestamps.diff().dropna().nunique() != 1:
        raise ValueError('timestamps must be regularly spaced for this tutorial path')
    series = frame[variate_names].to_numpy(dtype=float).T
    sample_kind = 'BYOD'
else:
    rng = np.random.default_rng(11)
    t = np.arange(320)
    first = 0.01 * t + np.sin(t / 9) + rng.normal(0, 0.04, len(t))
    second = 0.5 * first + np.cos(t / 13) + rng.normal(0, 0.04, len(t))
    series = np.vstack([first, second])
    variate_names = ['synthetic-trend-season', 'synthetic-coupled-season']
    sample_name = 'synthetic_two_variate_320.csv'
    sample_kind = 'synthetic'

series = validate_target(series)
if series.shape[1] < HORIZON + MIN_CONTEXT:
    raise ValueError(f'this tutorial needs at least {HORIZON + MIN_CONTEXT} rows ({MIN_CONTEXT} context + {HORIZON} holdout); got {series.shape[1]}')
sample_sha256 = hashlib.sha256(np.ascontiguousarray(series, dtype=np.float32).tobytes()).hexdigest()
print({'sample_kind': sample_kind, 'name': sample_name, 'shape': list(series.shape), 'horizon': HORIZON, 'decode_block_size': DECODE_BLOCK_SIZE, 'float32_sha256': sample_sha256})

## 5. Chronological holdout, then validate → input manifest

The final `HORIZON` steps are **withheld** as the truth; only the earlier context is passed to the model, so no future target value leaks into the forecast. `validate_inputs` is the pipeline's public validation stage: it applies the model-independent checks `forecast` applies — target shape and finiteness, context length `MIN_CONTEXT`..`MAX_CONTEXT`, horizon 1..`MAX_HORIZON`, the `decode_block_size` type — and returns an **input manifest** naming the schema and ceilings, each variate's observed context length and value range, and the verdict; the two patch-size rules (context ≥ 32, `decode_block_size` a multiple of 32) are read from the loaded model config and enforced by `forecast` itself. The manifest is written to `outputs/toto_forecasting_input_manifest.json`. To show what rejection looks like, the cell also validates a deliberately too-short context and records the pipeline's own error message as a finding. The naive **last-value baseline** (repeat the final observed value across the horizon) is computed here from the same context.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MIN_CONTEXT': MIN_CONTEXT, 'MAX_CONTEXT': MAX_CONTEXT, 'MAX_HORIZON': MAX_HORIZON}})
context = series[:, :-HORIZON]
truth = series[:, -HORIZON:]
input_manifest = validate_inputs(context, horizon=HORIZON, decode_block_size=DECODE_BLOCK_SIZE, names=variate_names)
# Demonstrate rejection on an input that breaks a ceiling; the finding is recorded, not swallowed.
try:
    validate_inputs(np.ones(MIN_CONTEXT - 1), horizon=HORIZON)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'short-context-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/toto_forecasting_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
baseline = last_value_baseline(context, HORIZON)
print(json.dumps(input_manifest, indent=2))
print({'context_length': context.shape[1], 'horizon': HORIZON, 'baseline_mae': mae(truth, baseline), 'baseline_rmse': rmse(truth, baseline)})

## 6. Forecast

`forecast` returns `quantiles` of shape `(variates, 9, horizon)` at `quantile_levels` 0.1 … 0.9, and `median` — the q=0.5 slice, which is the **point forecast** (`point_forecast` in the result). The median is a model median, not a mean, and the other quantiles are model quantiles rather than guaranteed confidence intervals; nothing is calibrated here. Upstream consumes the context in 32-step patches, so the pipeline left-pads the 272-step tutorial context to 288 with masked (unobserved) positions and reports the applied `context_padding` and `patch_size`; `decode_block_size=768` decodes the horizon in one block. The horizon and the effective context length are echoed in the result. Inference is zero-shot and deterministic given the same weights, device and library versions. Look for the first steps of each variate's median next to the withheld truth.

In [ ]:
result = pipe.forecast(context, horizon=HORIZON, decode_block_size=DECODE_BLOCK_SIZE)
prediction = result['median']
print({'point_forecast': result['point_forecast'], 'quantile_levels': list(result['quantile_levels']), 'horizon': result['horizon'], 'context_length': result['context_length'], 'context_padding': result['context_padding'], 'patch_size': result['patch_size'], 'decode_block_size': result['decode_block_size'], 'n_variates': result['n_variates'], 'device': result['device'], 'source': result['source']})
for variate in range(prediction.shape[0]):
    for step in range(min(3, HORIZON)):
        print(f"{variate_names[variate]} step {step + 1:>2}  median {prediction[variate, step]:.4f}  truth {truth[variate, step]:.4f}  q10 {result['quantiles'][variate, 0, step]:.4f}  q90 {result['quantiles'][variate, 8, step]:.4f}")

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. Because the truth was withheld chronologically in Section 5, it carries the repository's own `mae` and `rmse` on the median, the empirical `interval_coverage` of the q10–q90 band (nominal 0.8), and the same `mae`/`rmse` for the `last_value_baseline`, with the verdict `sample-sanity` — one holdout on the tutorial sample with no dispersion estimate, not a benchmark. Without withheld truth the verdict is `not-measurable` and the report states what would make the task measurable (a chronological holdout repeated over representative periods). The report is written to `outputs/toto_forecasting_evaluation_report.json`.

In [ ]:
report = evaluation_report(result, truth, context=context, sample_kind=sample_kind)
with open('outputs/toto_forecasting_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
if report['verdict'] == 'not-measurable':
    print('No withheld truth was supplied, so mae/rmse are not computed; the forecast above is sanity evidence only.')

## 8. Export outputs and provenance

Machine-readable JSON preserves the forecast summary (point-forecast semantics, quantile levels, horizon, context length, padding, patch size, decode strategy), the evaluation report, the input manifest, the sample identity and digest, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, and the runtime identity (Python, `torch`, `numpy`, `pandas`, device). The CSV keeps time step, variate, median, truth, last-value baseline and all nine quantiles aligned row by row. No credentials are recorded.

In [ ]:
rows = []
for variate in range(prediction.shape[0]):
    for step in range(HORIZON):
        row = {'variate': variate_names[variate], 'step': step + 1, 'median': float(prediction[variate, step]), 'truth': float(truth[variate, step]), 'last_value_baseline': float(baseline[variate, step])}
        for index, level in enumerate(result['quantile_levels']):
            row[f'q{int(round(level * 100)):02d}'] = float(result['quantiles'][variate, index, step])
        rows.append(row)
pd.DataFrame(rows).to_csv('outputs/toto_forecasting_forecast.csv', index=False)
payload = {
    'forecast': {'point_forecast': result['point_forecast'], 'quantile_levels': list(result['quantile_levels']), 'horizon': result['horizon'], 'context_length': result['context_length'], 'context_padding': result['context_padding'], 'patch_size': result['patch_size'], 'decode_block_size': result['decode_block_size'], 'n_variates': result['n_variates'], 'median': prediction.tolist()},
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'name': sample_name, 'shape': list(series.shape), 'float32_sha256': sample_sha256},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'numpy': numpy.__version__,
        'pandas': pandas.__version__,
        'device': pipe.device,
    },
}
with open('outputs/toto_forecasting_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The forecast is zero-shot; no gradient training or fine-tuning occurs. q=0.5 is a model median, and the other quantiles are model quantiles rather than guaranteed confidence intervals — the empirical q10–q90 coverage on one holdout is not a calibration statement. MAE/RMSE come from one chronological tutorial holdout of a synthetic series and must be repeated over representative periods of the real deployment series before any conclusion; the last-value baseline is the floor a useful forecaster must beat on that series, not a benchmark. Toto 2.0 exogenous-variable support and fine-tuning are deliberately excluded because they are not part of the current upstream 2.0 inference release; missing values are rejected rather than imputed; regime changes, irregular sampling and horizons far beyond the context all degrade results in ways the pipeline does not detect.

Successful execution proves that the recorded repository revision's package, carried in this notebook, can acquire and digest-verify the pinned checkpoint, validate the demonstrated input, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** enable `USE_BYOD` with a CSV from your own domain and compare the median's MAE against the last-value baseline over several consecutive holdouts (roll `HORIZON` forward); set `DECODE_BLOCK_SIZE = None` for a single forward-pass decode and compare the quantiles; shorten the context toward `MIN_CONTEXT` and watch the padding and the q10–q90 band change.

## References

- Repository README: https://github.com/kurtvalcorza/toto-forecasting-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/toto-forecasting-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/toto-forecasting-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/Datadog/Toto-2.0-2.5B
- Upstream code: https://github.com/DataDog/toto
- Technical report: https://arxiv.org/abs/2605.20119